# QSAR Aromatase â€” 16 Models Ã— 12 Fingerprints Ã— 2 Splits (GPU Accelerated)

**Target**: pchembl_value regression for Aromatase (CYP19A1) inhibitors  
**Total runs**: 16 Ã— 12 Ã— 2 = 384 model fits + 10-fold CV  
**GPU**: cuML + XGBoost CUDA | **Fallback**: sklearn CPU  

### How to run on Colab:
1. Upload `data.zip` to your Google Drive root folder
2. Open this notebook on colab.research.google.com
3. Runtime â†’ Change runtime type â†’ **GPU** (T4 recommended)
4. Run all cells (Ctrl+F9)


## 1. Download Data from GitHub


In [ ]:
# Download data from GitHub repo (no Drive/upload needed)
import os
if not os.path.exists("data"):
    !wget -q "https://github.com/dom-castaneda/qsar-aromatase/raw/master/data.zip" -O data.zip
    !unzip -qo data.zip
    print("Data extracted from GitHub.")
else:
    print("Data already present.")

assert os.path.exists("data/processed/aromatase_bioactivity_clean.csv")
assert os.path.exists("data/fingerprints_reduced/fingerprints_maccs.csv")
assert os.path.exists("data/splits/random_train.csv")
print("All data files verified.")


## 2. Install cuML & XGBoost

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkg.split())

install("xgboost")

try:
    install("--extra-index-url=https://pypi.nvidia.com cuml-cu12")
    print("cuML (cu12) installed")
except Exception:
    try:
        install("--extra-index-url=https://pypi.nvidia.com cuml-cu11")
        print("cuML (cu11) installed")
    except Exception:
        print("cuML install failed - will use sklearn")


## 3. Imports with GPU/CPU Fallback

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, time
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold, cross_val_predict

# --- GPU Detection with VALIDATION ---
USE_GPU = False
try:
    import cuml
    from cuml.linear_model import Ridge as cuRidge, Lasso as cuLasso, ElasticNet as cuElasticNet
    from cuml.neighbors import KNeighborsRegressor as cuKNN
    from cuml.svm import SVR as cuSVR
    from cuml.ensemble import RandomForestRegressor as cuRF
    # VALIDATE GPU actually works (catches driver mismatch)
    import cupy as cp
    cp.zeros(10)
    _test_model = cuRidge(alpha=1.0)
    _test_model.fit(cp.zeros((10, 3), dtype=cp.float32), cp.zeros(10, dtype=cp.float32))
    del _test_model
    USE_GPU = True
    print(f"GPU VALIDATED - using cuML {cuml.__version__}")
except Exception as e:
    USE_GPU = False
    print(f"GPU not available ({type(e).__name__}): {e}")
    print("Using sklearn (CPU) for all models")

from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.cross_decomposition import PLSRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, AdaBoostRegressor,
                              HistGradientBoostingRegressor)
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

XGB_DEVICE = "cuda" if USE_GPU else "cpu"
print(f"\nXGBoost device: {XGB_DEVICE}")
print(f"Mode: {'GPU (cuML + XGBoost CUDA)' if USE_GPU else 'CPU (sklearn + XGBoost CPU)'}")


## 4. Load Data

In [ ]:
RANDOM_STATE = 42
N_FOLDS = 10
BASE = "/content/data"

FP_NAMES = {
    "MACCS": "fingerprints_maccs.csv",
    "PubChem": "fingerprints_pubchem.csv",
    "Substructure": "fingerprints_substruct.csv",
    "SubstructureCount": "fingerprints_substruct_count.csv",
    "KR": "fingerprints_kr.csv",
    "KR_Count": "fingerprints_kr_count.csv",
    "AtomPairs2D": "fingerprints_atompairs2d.csv",
    "AP2D_Count": "fingerprints_atompairs2d_count.csv",
    "CDK_FP": "fingerprints_cdk_fp.csv",
    "CDK_Extended": "fingerprints_cdk_extended.csv",
    "CDK_GraphOnly": "fingerprints_cdk_graphonly.csv",
    "EState": "fingerprints_estate.csv",
    "EState_Count": "fingerprints_estate_count.csv",
    "ECFP4": "fingerprints_ecfp4.csv",
}

SPLITS = {
    "Random": (f"{BASE}/splits/random_train.csv", f"{BASE}/splits/random_test.csv"),
    "Kennard-Stone": (f"{BASE}/splits/kennard_stone_train.csv", f"{BASE}/splits/kennard_stone_test.csv"),
}

df_full = pd.read_csv(f"{BASE}/processed/aromatase_bioactivity_clean.csv")
mask = (df_full["standard_relation"] == "=") & df_full["pchembl_value"].notna()
df = df_full[mask].reset_index(drop=True)
print(f"Dataset: {len(df)} molecules")

# Load fingerprints â€” fill NaN with 0 (PubChem has 22 missing molecules)
fp_data = {}
for fp_name, fp_file in FP_NAMES.items():
    fp_full = pd.read_csv(f"{BASE}/fingerprints_reduced/{fp_file}")
    fp_filtered = fp_full[mask.values].reset_index(drop=True)
    fp_cols = [c for c in fp_filtered.columns if c != "molecule_chembl_id"]
    X = fp_filtered[fp_cols].values.astype(np.float32)
    # Fill NaN with 0 (affects PubChem where some molecules not found)
    nan_count = np.isnan(X).sum()
    if nan_count > 0:
        X = np.nan_to_num(X, nan=0.0)
        print(f"  {fp_name:<20} {X.shape[1]:>5} features (filled {nan_count} NaN -> 0)")
    else:
        print(f"  {fp_name:<20} {X.shape[1]:>5} features")
    fp_data[fp_name] = X

# Load splits
split_masks = {}
for split_name, (train_file, test_file) in SPLITS.items():
    train_ids = set(pd.read_csv(train_file)["molecule_chembl_id"])
    test_ids = set(pd.read_csv(test_file)["molecule_chembl_id"])
    train_mask = df["molecule_chembl_id"].isin(train_ids).values
    test_mask = df["molecule_chembl_id"].isin(test_ids).values
    split_masks[split_name] = (train_mask, test_mask)
    print(f"  {split_name}: train={train_mask.sum()}, test={test_mask.sum()}")

y_all = df["pchembl_value"].values
print(f"\nReady: {len(FP_NAMES)} FPs x {len(SPLITS)} splits x 16 models = {len(FP_NAMES)*len(SPLITS)*16} runs")


## 5. Model Builder

In [ ]:
def build_models():
    models = []
    if USE_GPU:
        models.append(("Ridge", cuRidge(alpha=1.0)))
        models.append(("Lasso", cuLasso(alpha=0.1)))
        models.append(("ElasticNet", cuElasticNet(alpha=0.1, l1_ratio=0.5)))
    else:
        models.append(("Ridge", Ridge(alpha=1.0)))
        models.append(("Lasso", Lasso(alpha=0.1, max_iter=5000)))
        models.append(("ElasticNet", ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000)))

    models.append(("Bayesian Ridge", BayesianRidge()))
    models.append(("PLS", PLSRegression(n_components=10)))

    if USE_GPU:
        models.append(("KNN", cuKNN(n_neighbors=5)))
    else:
        models.append(("KNN", KNeighborsRegressor(n_neighbors=5)))

    if USE_GPU:
        models.append(("SVR (RBF)", cuSVR(kernel="rbf", C=1.0, epsilon=0.1)))
    else:
        models.append(("SVR (RBF)", SVR(kernel="rbf", C=1.0, epsilon=0.1)))

    models.append(("Kernel Ridge (RBF)", KernelRidge(alpha=1.0, kernel="rbf")))
    models.append(("Decision Tree", DecisionTreeRegressor(random_state=RANDOM_STATE)))

    if USE_GPU:
        models.append(("Random Forest", cuRF(n_estimators=500, random_state=RANDOM_STATE)))
    else:
        models.append(("Random Forest", RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)))

    models.append(("Extra Trees", ExtraTreesRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)))
    models.append(("Gradient Boosting", GradientBoostingRegressor(n_estimators=500, random_state=RANDOM_STATE)))
    models.append(("XGBoost", XGBRegressor(n_estimators=500, learning_rate=0.1,
                                            random_state=RANDOM_STATE, verbosity=0,
                                            device=XGB_DEVICE, n_jobs=-1)))
    models.append(("Hist Gradient Boosting", HistGradientBoostingRegressor(max_iter=500, random_state=RANDOM_STATE)))
    models.append(("AdaBoost", AdaBoostRegressor(n_estimators=500, random_state=RANDOM_STATE)))
    models.append(("MLP", MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=500,
                                        random_state=RANDOM_STATE, early_stopping=True)))
    return models

def compute_metrics(y_true, y_pred):
    return r2_score(y_true, y_pred), np.sqrt(mean_squared_error(y_true, y_pred)), mean_absolute_error(y_true, y_pred)

print(f"{len(build_models())} models ready. GPU={USE_GPU}")


## 6. Training Loop

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
all_results = []
total_runs = len(SPLITS) * len(FP_NAMES) * 16
run_count = 0
t_start = time.time()

for split_name, (train_mask, test_mask) in split_masks.items():
    y_train = y_all[train_mask]
    y_test = y_all[test_mask]

    for fp_name, X_fp in fp_data.items():
        X_train = X_fp[train_mask]
        X_test = X_fp[test_mask]
        models = build_models()

        for model_name, model in models:
            run_count += 1
            t0 = time.time()

            try:
                # 10-fold CV
                try:
                    y_cv_pred = cross_val_predict(model, X_train, y_train, cv=kf, n_jobs=1)
                    r2_cv, rmse_cv, mae_cv = compute_metrics(y_train, y_cv_pred)
                except Exception:
                    r2_cv, rmse_cv, mae_cv = np.nan, np.nan, np.nan

                model.fit(X_train, y_train)
                y_train_pred = np.asarray(model.predict(X_train)).ravel()
                y_test_pred = np.asarray(model.predict(X_test)).ravel()
                r2_train, rmse_train, mae_train = compute_metrics(y_train, y_train_pred)
                r2_test, rmse_test, mae_test = compute_metrics(y_test, y_test_pred)
                elapsed = time.time() - t0

                all_results.append({
                    "Split": split_name, "Fingerprint": fp_name, "Model": model_name,
                    "R2_train": r2_train, "RMSE_train": rmse_train, "MAE_train": mae_train,
                    "R2_CV": r2_cv, "RMSE_CV": rmse_cv, "MAE_CV": mae_cv,
                    "R2_test": r2_test, "RMSE_test": rmse_test, "MAE_test": mae_test,
                    "Time_s": elapsed,
                })

            except Exception as e:
                elapsed = time.time() - t0
                all_results.append({
                    "Split": split_name, "Fingerprint": fp_name, "Model": model_name,
                    "R2_train": np.nan, "RMSE_train": np.nan, "MAE_train": np.nan,
                    "R2_CV": np.nan, "RMSE_CV": np.nan, "MAE_CV": np.nan,
                    "R2_test": np.nan, "RMSE_test": np.nan, "MAE_test": np.nan,
                    "Time_s": elapsed,
                })
                print(f"  FAILED: {split_name}/{fp_name}/{model_name}: {e}")

            if True:
                elapsed_total = time.time() - t_start
                eta = (elapsed_total / run_count) * (total_runs - run_count)
                print(f"  [{run_count}/{total_runs}] {split_name}|{fp_name}|{model_name} "
                      f"R2={all_results[-1]['R2_test']:.4f} ({elapsed_total:.0f}s, ETA ~{eta/60:.0f}min)")

total_time = time.time() - t_start
print(f"\nDONE: {run_count} runs in {total_time:.1f}s ({total_time/60:.1f} min)")


## 7. Save Results

In [ ]:
results_df = pd.DataFrame(all_results)

# Save locally and to Drive
results_df.to_csv("/content/results_all_models_reduced.csv", index=False)
try:
    results_df.to_csv("/content/drive/My Drive/results_all_models_reduced.csv", index=False)
    print("Saved to Google Drive")
except:
    pass

print(f"results_all_models_reduced.csv: {len(results_df)} rows x {len(results_df.columns)} columns")
n_failed = results_df["R2_test"].isna().sum()
print(f"Successful: {len(results_df) - n_failed}/{len(results_df)}, Failed: {n_failed}")
results_df.sort_values("R2_test", ascending=False).head(20)


## 8. Best Models Summary

In [ ]:
print("=" * 90)
print(f"{'BEST MODEL PER FINGERPRINT':^90}")
print("=" * 90)

for split_name in SPLITS:
    print(f"\n--- {split_name} Split ---")
    sub = results_df[(results_df["Split"] == split_name) & results_df["R2_test"].notna()]
    if len(sub) == 0:
        print("  No successful runs")
        continue
    best_per_fp = sub.loc[sub.groupby("Fingerprint")["R2_test"].idxmax()]
    best_per_fp = best_per_fp.sort_values("R2_test", ascending=False)
    print(f"{'Fingerprint':<20} {'Model':<25} {'R2_test':<9} {'RMSE_test':<10} {'MAE_test':<9}")
    print("-" * 75)
    for _, row in best_per_fp.iterrows():
        print(f"{row['Fingerprint']:<20} {row['Model']:<25} {row['R2_test']:<9.4f} "
              f"{row['RMSE_test']:<10.4f} {row['MAE_test']:<9.4f}")

valid = results_df.dropna(subset=["R2_test"])
if len(valid) > 0:
    best = valid.loc[valid["R2_test"].idxmax()]
    print(f"\nOVERALL BEST: {best['Model']} on {best['Fingerprint']} ({best['Split']})")
    print(f"  R2={best['R2_test']:.4f}, RMSE={best['RMSE_test']:.4f}, MAE={best['MAE_test']:.4f}")


## 9. Heatmap â€” Test RÂ²

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for split_name in SPLITS:
    sub = results_df[(results_df["Split"] == split_name) & results_df["R2_test"].notna()]
    if len(sub) == 0:
        continue
    pivot = sub.pivot_table(index="Model", columns="Fingerprint", values="R2_test")
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=0.7,
                linewidths=0.5, ax=ax, cbar_kws={"label": "R2 (test)"})
    ax.set_title(f"Test R2 - {split_name} Split", fontsize=14)
    plt.tight_layout()
    fname = f"/content/heatmap_{split_name.lower().replace('-','_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    try:
        plt.savefig(f"/content/drive/My Drive/heatmap_{split_name.lower().replace('-','_')}.png",
                    dpi=150, bbox_inches="tight")
    except:
        pass


## 10. Download

In [ ]:
from google.colab import files
files.download("/content/results_all_models_reduced.csv")
print("Results also saved to Google Drive.")
